# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [41]:
from sklearn.datasets import  fetch_california_housing
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
# New in here:
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

In [15]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [16]:
df_spaceship = spaceship.dropna()
df_spaceship

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False


In [17]:
has_null_cabin = df_spaceship['Cabin'].isnull().any()
df_spaceship['Cabin'] = df_spaceship['Cabin'].map(lambda x: x[0])

C:\Users\leolo\AppData\Local\Temp\ipykernel_30612\3941257211.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_spaceship['Cabin'] = df_spaceship['Cabin'].map(lambda x: x[0])


In [18]:
print(df_spaceship['Cabin'].unique())

['B' 'F' 'A' 'G' 'E' 'C' 'D' 'T']


In [19]:
df_cleaned = df_spaceship.drop(columns=['PassengerId', 'Name'])

In [20]:
non_numerical_columns = df_cleaned.select_dtypes(exclude=['number']).columns
non_numerical_columns

Index(['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP',
       'Transported'],
      dtype='object')

In [21]:
non_numerical_columns = pd.get_dummies(df_cleaned, columns=non_numerical_columns, drop_first=True, dtype=int)
non_numerical_columns

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True,Transported_True
0,39.0,0.0,0.0,0.0,0.0,0.0,1,0,0,1,0,0,0,0,0,0,0,1,0,0
1,24.0,109.0,9.0,25.0,549.0,44.0,0,0,0,0,0,0,0,1,0,0,0,1,0,1
2,58.0,43.0,3576.0,0.0,6715.0,49.0,1,0,0,0,0,0,0,0,0,0,0,1,1,0
3,33.0,0.0,1283.0,371.0,3329.0,193.0,1,0,0,0,0,0,0,0,0,0,0,1,0,0
4,16.0,303.0,70.0,151.0,565.0,2.0,0,0,0,0,0,0,0,1,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,41.0,0.0,6819.0,0.0,1643.0,74.0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
8689,18.0,0.0,0.0,0.0,0.0,0.0,0,0,1,0,0,0,0,0,1,0,1,0,0,0
8690,26.0,0.0,0.0,1872.0,1.0,0.0,0,0,0,0,0,0,0,0,1,0,0,1,0,1
8691,32.0,0.0,1049.0,0.0,353.0,3235.0,1,0,0,0,0,0,1,0,0,0,0,0,0,0


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [22]:
features = non_numerical_columns.drop(columns=['Transported_True'])
features

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,1,0,0,1,0,0,0,0,0,0,0,1,0
1,24.0,109.0,9.0,25.0,549.0,44.0,0,0,0,0,0,0,0,1,0,0,0,1,0
2,58.0,43.0,3576.0,0.0,6715.0,49.0,1,0,0,0,0,0,0,0,0,0,0,1,1
3,33.0,0.0,1283.0,371.0,3329.0,193.0,1,0,0,0,0,0,0,0,0,0,0,1,0
4,16.0,303.0,70.0,151.0,565.0,2.0,0,0,0,0,0,0,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,41.0,0.0,6819.0,0.0,1643.0,74.0,1,0,0,0,0,0,0,0,0,0,0,0,1
8689,18.0,0.0,0.0,0.0,0.0,0.0,0,0,1,0,0,0,0,0,1,0,1,0,0
8690,26.0,0.0,0.0,1872.0,1.0,0.0,0,0,0,0,0,0,0,0,1,0,0,1,0
8691,32.0,0.0,1049.0,0.0,353.0,3235.0,1,0,0,0,0,0,1,0,0,0,0,0,0


In [23]:
target = non_numerical_columns["Transported_True"]
target

0       0
1       1
2       0
3       0
4       1
       ..
8688    0
8689    0
8690    1
8691    0
8692    1
Name: Transported_True, Length: 6606, dtype: int64

In [24]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size= 0.20 , random_state =0)
X_train.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
3432,32.0,0.0,0.0,0.0,0.0,0.0,0,1,1,0,0,0,1,0,0,0,0,1,0
7312,4.0,0.0,0.0,0.0,0.0,0.0,0,0,1,0,0,0,0,0,1,0,0,1,0
2042,30.0,0.0,236.0,0.0,1149.0,0.0,0,0,0,0,0,0,0,1,0,0,0,1,0
4999,17.0,13.0,0.0,565.0,367.0,1.0,0,1,0,0,0,0,1,0,0,0,0,1,0
5755,26.0,0.0,0.0,0.0,0.0,0.0,0,0,1,0,0,0,0,0,1,0,0,1,0


In [25]:
normalizer = MinMaxScaler()

In [29]:
normalizer.fit(X_train)

,feature_range,"(0, ...)"
,copy,True
,clip,False


In [35]:
X_train_norm = normalizer.transform(X_train)

In [36]:
X_train_norm

array([[4.05063291e-01, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 1.00000000e+00, 0.00000000e+00],
       [5.06329114e-02, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 1.00000000e+00, 0.00000000e+00],
       [3.79746835e-01, 0.00000000e+00, 7.91600979e-03, ...,
        0.00000000e+00, 1.00000000e+00, 0.00000000e+00],
       ...,
       [4.55696203e-01, 0.00000000e+00, 1.59527723e-01, ...,
        0.00000000e+00, 1.00000000e+00, 0.00000000e+00],
       [4.30379747e-01, 0.00000000e+00, 1.34169658e-04, ...,
        0.00000000e+00, 1.00000000e+00, 1.00000000e+00],
       [1.77215190e-01, 2.01612903e-04, 2.95508671e-02, ...,
        0.00000000e+00, 1.00000000e+00, 0.00000000e+00]], shape=(5284, 19))

In [ ]:
X_train_norm = normalizer.transform(X_train)
X_train_norm = pd.DataFrame(X_train_norm, columns = X_train.columns)
X_train_norm.head(3)

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,0.405063,0.0,0.000000,0.0,0.000000,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.050633,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0.379747,0.0,0.007916,0.0,0.051276,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [ ]:
X_test_norm = normalizer.transform(X_test)
X_test_norm = pd.DataFrame(X_test_norm, columns = X_test.columns)

X_test_norm.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,0.632911,0.0,0.000000,0.0,0.00000,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,0.227848,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
2,0.189873,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.658228,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,0.784810,0.0,0.054775,0.0,0.07774,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [37]:
bagging_reg = BaggingRegressor(DecisionTreeRegressor(max_depth = 20),
                             n_estimators = 100,
                             max_samples = 2000,
                             bootstrap = True)

bagging_reg.fit (X_train_norm, y_train)

,estimator,DecisionTreeR...(max_depth=20)
,n_estimators,100
,max_samples,2000
,max_features,1.0
,bootstrap,True
,bootstrap_features,False
,oob_score,False
,warm_start,False
,n_jobs,None
,random_state,None
,verbose,0


In [38]:
pred = bagging_reg.predict(X_test_norm)

c:\Users\leolo\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but BaggingRegressor was fitted without feature names
  warnings.warn(


In [39]:
print(f"R2 score: ", r2_score (y_test, pred))
print(f"RMSE: ", root_mean_squared_error (y_test, pred))
print(f"MAE: ", mean_absolute_error (y_test, pred))

R2 score:  0.41763099015480976
RMSE:  0.3815655283975448
MAE:  0.2752686405898975


- Evaluate your model

In [ ]:
#Bagging/Pasting achieved the best overall performance, with the highest R² score and the lowest RMSE among the tested models.

**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [40]:
gb_reg = GradientBoostingRegressor()

In [44]:
grid = {"n_estimators": [50, 100, 150, 200],
       "max_leaf_nodes": [50, 150, 300],
		"max_depth":[5, 15, 30]}

In [45]:
model = GridSearchCV(estimator = gb_reg, param_grid = grid, cv = 5, n_jobs = -1)

In [46]:
model.fit(X_train_norm, y_train)

,estimator,GradientBoostingRegressor()
,param_grid,"{'max_depth': [5, 15, ...], 'max_leaf_nodes': [50, 150, ...], 'n_estimators': [50, 100, ...]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'squared_error'


- Run Grid Search

In [52]:
model.best_params_

{'max_depth': 5, 'max_leaf_nodes': 50, 'n_estimators': 50}

In [53]:
best_model = model.best_estimator_

In [54]:
best_model


,loss,'squared_error'
,learning_rate,0.1
,n_estimators,50
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,5
,min_impurity_decrease,0.0
,init,None


In [55]:
pred = best_model.predict(X_test_norm)

c:\Users\leolo\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but GradientBoostingRegressor was fitted without feature names
  warnings.warn(


In [56]:
print(f"R2 score: ", r2_score (y_test, pred))
print(f"RMSE: ", root_mean_squared_error (y_test, pred))
print(f"MAE: ", mean_absolute_error (y_test, pred))

R2 score:  0.4301522235709888
RMSE:  0.37744131213640725
MAE:  0.2827461261600591


In [ ]:
#Random

In [57]:
ada_reg = AdaBoostRegressor(DecisionTreeRegressor())

In [58]:
grid_5 = {"n_estimators": np.arange(5,100),
        "estimator__max_leaf_nodes": np.arange(5,50),
		"estimator__max_depth": np.arange(3,50)}

In [59]:
model_5 = RandomizedSearchCV(estimator = ada_reg, param_distributions = grid_5, n_iter = 20, cv = 5, n_jobs = -1)

In [60]:
model_5.fit(X_train_norm,y_train)

,estimator,AdaBoostRegre...eeRegressor())
,param_distributions,"{'estimator__max_depth': array([ 3, 4..., 47, 48, 49]), 'estimator__max_leaf_nodes': array([ 5, 6..., 47, 48, 49]), 'n_estimators': array([ 5, 6..., 97, 98, 99])}"
,n_iter,20
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [61]:
model_5.best_params_

{'n_estimators': np.int64(5),
 'estimator__max_leaf_nodes': np.int64(37),
 'estimator__max_depth': np.int64(21)}

In [62]:
best_model_5 = model_5.best_estimator_

In [63]:
pred = best_model_5.predict(X_test_norm)

c:\Users\leolo\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but AdaBoostRegressor was fitted without feature names
  warnings.warn(


In [64]:
print(f"R2 score: ", r2_score (y_test, pred))
print(f"RMSE: ", root_mean_squared_error (y_test, pred))
print(f"MAE: ", mean_absolute_error (y_test, pred))

R2 score:  0.41574994080405114
RMSE:  0.38218125908917516
MAE:  0.29453667773837383


- Evaluate your model

In [ ]:
#Grid Search achieved the best overall performance, with the highest R² score and the lowest RMSE.